---
title: "4.1 Descriptive statistics and a Table 1"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/04-regression/01-descriptive-statistics.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/04-regression/01-descriptive-statistics.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [3.4 Saving figures for a paper or slides](../03-viz/04-saving-figures.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Report counts, centre and spread with their measurement units.
- Build a descriptive Table 1 with explicit denominators.
- Keep simulated people separate from historical country observations.

## Why it matters

A **descriptive statistic** summarizes observed measurements. A **Table 1** is an introductory table describing an analysis sample; the name does not prescribe one universal format. Before fitting a model, show who or what was observed, what is missing, and what units are being summarized. We start with made-up wages, describe the real country cross-section, then build a table from the simulated person-level thread.

## Worked example

### Compare centre and spread

These four hourly wages are made-up numbers for fictional Northland respondents. The **mean** divides their sum by the count; the **median** is the middle ordered value (or average of two middle values). The **standard deviation** summarizes spread around the mean. pandas' std uses the sample convention, dividing squared deviations by n−1 before taking a square root.

In [ ]:
import pandas as pd
small = pd.DataFrame({"respondent": ["Northland A", "Northland B", "Northland C", "Northland D"], "hourly_wage": [10.0, 12.0, 14.0, 100.0]})
small_summary = small["hourly_wage"].agg(["count", "mean", "median", "std", "min", "max"])
print(small_summary.round(2))

The mean is 34 dollars per hour, while the median is 13. The 100-dollar value pulls the mean upward. A standard deviation describes spread; it is not the standard error of an estimated mean or regression coefficient. Check whether an extreme value is real before choosing a method or deleting it.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Describe a real cross-section

Select 2007 before calculating country-level summaries. `.describe()` reports non-missing count, mean, sample standard deviation, range and quartiles (values marking the lower quarter, middle and upper quarter of the ordered data).

In [ ]:
countries = pd.read_csv(data_file)
year_2007 = countries.loc[countries["year"] == 2007].copy()
country_description = year_2007[["life_exp", "gdp_pc", "population"]].describe()
country_description.round(2)

All three counts are 142 here. Each country has equal weight. Life expectancy is in years, GDP per person in the extract's inflation-adjusted dollars and population in people. Combining them in a table does not give them the same units. A country-weighted mean does not describe the average person in the world.

### Prepare the simulated survey

Run the shared, fixed-seed simulation setup. Its 120 records describe fictional people, not an actual population. The exact generator and missing-value conventions are in the [data documentation](https://github.com/ningzhanglab/Python-for-Beginners/blob/main/data/README.md). A codebook is a guide to a dataset's variable definitions, units and coding rules.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared simulated-data setup (run this cell)"
# Prepared setup: simulated people, generated locally; no real survey data.
from pathlib import Path
import csv
import io
import random

def generate_people():
    """Return a deterministic UTF-8 CSV using only Python's standard library."""
    rng = random.Random(20261111)
    output = io.StringIO(newline="")
    writer = csv.writer(output, lineterminator="\n")
    writer.writerow(["person_id", "age", "education", "education_years", "employed", "hourly_wage"])
    labels = {12: "secondary", 14: "college", 16: "university", 18: "postgraduate"}
    for person_id in range(1, 121):
        age = rng.randint(22, 64)
        education_years = rng.choice([12, 14, 16, 18])
        employed = int(rng.random() >= 0.2)
        wage = round(8 + 0.9 * education_years + 0.08 * age + rng.uniform(-2, 2), 2)
        # Blank wages for non-workers; -99 means a worker did not report a wage.
        wage_text = str(wage) if employed else ""
        if employed and person_id % 7 == 0:
            wage_text = "-99"
        education = labels[education_years]
        if person_id % 9 == 0:
            education = " " + education.upper() + " "
        age_text = str(age) if person_id % 13 else "unknown"
        writer.writerow([person_id, age_text, education, education_years, employed, wage_text])
    return output.getvalue().encode("utf-8")

people_file = Path("data/simulated_people.csv")
people_file.parent.mkdir(parents=True, exist_ok=True)
if not people_file.exists():
    people_file.write_bytes(generate_people())
print("Simulated data ready: 120 fictional people; not evidence about real wages.")

The setup produces the same simulated file in a fresh runtime. Blank wages mean non-employment and -99 means an employed person did not report a wage; neither is an observed zero wage. The simulated wage pattern is built into the generator, not empirical evidence.

### Define cleaned survey fields

Load the simulated records, preserve the raw fields and create cleaned age and wage columns as in lesson 2.4. The displayed counts track conversion failures and unavailable wages before any summary is computed.

In [ ]:
people = pd.read_csv(people_file, dtype={"age": "string"})
people["age_clean"] = pd.to_numeric(people["age"], errors="coerce").astype("Int64")
people["wage_clean"] = people["hourly_wage"].mask(people["hourly_wage"] == -99)
print("Simulated people:", len(people))
print("Missing clean ages:", people["age_clean"].isna().sum())
print("Missing clean wages:", people["wage_clean"].isna().sum())

The file has 120 fictional people, nine missing clean ages and 33 unavailable clean wages. Those 33 comprise 19 non-workers and 14 employed non-respondents. Setting them to zero would change the question and distort the wage summary.

### Build a Table 1 by employment status

Use `.agg` to name each summary and include its contributing count. `n_people` is the group size; age and wage counts may be smaller. The renamed group labels make the coding understandable without relying on 0/1 values alone.

In [ ]:
table1 = people.groupby("employed").agg(
    n_people=("person_id", "size"),
    age_available=("age_clean", "count"),
    mean_age_years=("age_clean", "mean"),
    wage_available=("wage_clean", "count"),
    mean_hourly_wage=("wage_clean", "mean"),
)
table1 = table1.rename(index={0: "Not employed", 1: "Employed"})
table1.round(2)

There are 19 non-workers and 101 workers. Only 87 workers contribute a wage. The non-worker wage mean is missing because no wage is defined for that group in this simulation; it is not a zero-dollar wage estimate. These are simulated, unweighted summaries with no real population or dollar base year.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Describe the real 2002 sample

Create country-level life-expectancy summary statistics for 2002. Compare the count, mean and median with 2007 without claiming a causal time effect.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
year_2002 = countries.loc[countries["year"] == 2002]
comparison = pd.DataFrame({"2002": year_2002["life_exp"].agg(["count", "mean", "median"]), "2007": year_2007["life_exp"].agg(["count", "mean", "median"])})
comparison.round(2)

Both samples contain 142 country observations. The means and medians differ, but a time comparison alone cannot distinguish income, health, composition or other explanations.

</details>

### 2. Reconcile the Table 1 denominators

Add the group counts and the available-wage counts. Which totals describe different sets of simulated people?

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
print("All group members:", table1["n_people"].sum())
print("Available wages:", table1["wage_available"].sum())

The totals are 120 and 87. Each mean must be read with the denominator for its own variable, not automatically with the full group size.

</details>

### 3. Report a robust descriptive centre

Calculate the median cleaned wage among simulated workers with a reported wage. Store it as `median_worker_wage` and print its unit.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
wage_sample = people.loc[(people["employed"] == 1) & people["wage_clean"].notna()].copy()
median_worker_wage = wage_sample["wage_clean"].median()
print("Simulated dollars per hour:", round(median_worker_wage, 2))

The median summarizes only the 87 simulated workers with reported wages. “Robust” here means less sensitive to extreme observations; it does not imply representative sampling or a causal wage estimate.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### The raw text is treated as a numeric age

One simulated raw response is the word unknown.

In [ ]:
try:
    people["age"].astype(float)
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Use the documented conversion into age_clean, count failures and retain the original responses. A conversion rule cannot establish whether an implausible recorded age is correct.

### The summary column name is guessed

A named aggregation creates the names you explicitly chose.

In [ ]:
try:
    table1["average_wage"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Inspect table1.columns. The actual field is mean_hourly_wage. Meaningful labels and a codebook are more reliable than guessing a common name.

## Recap

- Describe the sample before estimating a model.
- Mean, median, standard deviation and standard error have different meanings.
- Show available counts by variable and explain missingness.
- The simulated Table 1 is a teaching artifact, not a population description.

**Next:** [4.2 Correlation and simple OLS](../04-regression/02-correlation-ols.ipynb).